# Machine Learning for Compressive Strength Prediction of Nano-Silica Modified Concrete

This notebook develops a machine learning workflow for estimating concrete compressive strength from mixture proportions and curing age. The analysis focuses on nano-silica modified concrete and follows a complete regression workflow: dataset loading, exploratory analysis, data splitting, normalization, ANN training, and model evaluation.


## Imports and Configuration

Load the scientific Python libraries, configure project paths, and set random seeds for reproducible experiments.


In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data import TARGET_COLUMN, load_dataset
from src.evaluation import evaluate_regression
from src.models import build_ann_model, make_training_callbacks, run_hidden_unit_experiment
from src.preprocessing import normalize_features, split_dataset
from src.visualization import (
    plot_actual_vs_predicted,
    plot_correlation_heatmap,
    plot_feature_distributions,
    plot_prediction_errors,
    plot_training_history,
)

DATA_PATH = PROJECT_ROOT / "data" / "Nanodata.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass


## Project Modules

Reusable functions for data loading, preprocessing, model training, evaluation, and visualization live in `src/`. The notebook imports those functions above and uses them throughout the workflow.


## Dataset Description

The dataset contains concrete mixture designs and measured compressive strength values. Each row represents one concrete sample, and the columns describe binder content, water, supplementary cementitious materials, aggregates, admixture dosage, curing age, specimen size, and compressive strength.


In [ ]:
concrete_data = load_dataset(DATA_PATH)
concrete_data.head()


In [ ]:
print(f"Number of rows/examples and columns in the dataset: {concrete_data.shape}")
concrete_data.info()

In [ ]:
print("Display NA values in each columns: ")
concrete_data.isna().sum(axis=0)

In [ ]:
print("Display NA values in each row: ")
concrete_data.isna().sum(axis=1)

In [ ]:
print("Display NULL values in each columns: ")
concrete_data.isnull().sum()

## Feature and Target Variables

The prediction target is `compressive_strength`, measured in MPa. The remaining columns are used as input features that describe the mixture composition, age, and specimen properties. The original `strength` column is removed because `compressive_strength` is the target column used in this workflow.


## Exploratory Data Analysis

Explore feature distributions and correlations in the concrete mixture dataset. The plots are saved to `results/figures/` so they can be reused in the README or project report.


In [ ]:
plot_feature_distributions(concrete_data, output_dir=FIGURES_DIR)


In [ ]:
plot_correlation_heatmap(concrete_data, output_dir=FIGURES_DIR)


## Train, Validation, and Test Split

Split the data once into a fixed 60% training set, 20% validation set, and 20% test set. The training set is used to fit the model, the validation set is used to monitor training performance, and the test set is held out for final evaluation.


In [ ]:
train_dataset, validation_dataset, test_dataset = split_dataset(concrete_data)

print(f"Train dataset      : {train_dataset.shape}")
print(f"Validation dataset : {validation_dataset.shape}")
print(f"Test dataset       : {test_dataset.shape}")


## Data Normalization

Normalize the input features using statistics calculated from the training set. Applying the same normalization to validation and test data prevents information leakage from the held-out sets.


In [ ]:
normalized_data = normalize_features(train_dataset, validation_dataset, test_dataset)

train_features = normalized_data["train_features"]
validation_features = normalized_data["validation_features"]
test_features = normalized_data["test_features"]
train_labels = normalized_data["train_labels"]
validation_labels = normalized_data["validation_labels"]
test_labels = normalized_data["test_labels"]
normed_train_data = normalized_data["normed_train_data"]
normed_validation_data = normalized_data["normed_validation_data"]
normed_test_data = normalized_data["normed_test_data"]
feature_means = normalized_data["feature_means"]
feature_stds = normalized_data["feature_stds"]

print(f"Train features      : {train_features.shape}")
print(f"Validation features : {validation_features.shape}")
print(f"Test features       : {test_features.shape}")
print(f"Train labels        : {train_labels.shape}")
print(f"Validation labels   : {validation_labels.shape}")
print(f"Test labels         : {test_labels.shape}")


In [ ]:
train_features.describe().transpose()


In [ ]:
normed_train_data.head(10)


## Artificial Neural Network Model

Define a feed-forward artificial neural network for regression. The model receives the normalized mixture features as input and predicts compressive strength as a continuous output.


In [ ]:
model = build_ann_model(input_shape=normed_train_data.shape[1], hidden_units=52)
print("ANN model summary:")
model.summary()


## ANN Model Training

Train the neural network using the training split and monitor validation loss during training. The checkpoint callback stores the best model weights according to validation performance.


In [ ]:
%%time
MAX_EPOCHS = 5000
PATIENCE = 200
batch_size = 10
checkpoint_path = MODELS_DIR / "ProjectData.weights.h5"

callbacks = make_training_callbacks(
    checkpoint_path=checkpoint_path,
    patience=PATIENCE,
)

history = model.fit(
    normed_train_data,
    train_labels,
    batch_size=batch_size,
    epochs=MAX_EPOCHS,
    verbose=0,
    shuffle=True,
    validation_data=(normed_validation_data, validation_labels),
    callbacks=callbacks,
)

history_frame = pd.DataFrame(history.history)
history_frame["epoch"] = history.epoch
print(f"Training stopped after {len(history.epoch)} epochs.")
history_frame.tail()


## Training History

Plot the training and validation error curves to inspect convergence and check whether the model is underfitting or overfitting.


In [ ]:
history_frame = plot_training_history(history, output_dir=FIGURES_DIR)


## Model Evaluation

Evaluate the trained model on the training, validation, and test splits using a compact regression metrics table. MAE, RMSE, and R2 are treated as the primary metrics. MAPE is included as a secondary metric because percentage error can be less stable when measured strengths are small.


In [ ]:
train_predictions = model.predict(normed_train_data).flatten()
validation_predictions = model.predict(normed_validation_data).flatten()
test_predictions = model.predict(normed_test_data).flatten()

metrics = pd.DataFrame(
    [
        evaluate_regression(train_labels, train_predictions),
        evaluate_regression(validation_labels, validation_predictions),
        evaluate_regression(test_labels, test_predictions),
    ],
    index=["train", "validation", "test"],
)

primary_metrics = ["MAE", "RMSE", "R2"]
secondary_metrics = ["MAPE (%)"]
metrics = metrics[primary_metrics + secondary_metrics].round(3)
metrics


## Prediction Examples

Compare several predicted compressive strength values with their corresponding measured values from the training and test sets.


In [ ]:
prediction_examples = pd.DataFrame({
    "measured_strength": test_labels.head(10).to_numpy(),
    "predicted_strength": test_predictions[:10],
})
prediction_examples


## Prediction Plots

Visualize measured versus predicted compressive strength and inspect the prediction error distribution on the held-out test data.


In [ ]:
plot_actual_vs_predicted(train_labels, train_predictions, split_name="train", output_dir=FIGURES_DIR)


In [ ]:
plot_actual_vs_predicted(test_labels, test_predictions, split_name="test", output_dir=FIGURES_DIR)


In [ ]:
plot_prediction_errors(test_labels, test_predictions, split_name="test", output_dir=FIGURES_DIR)


## Automated ANN Hidden-Layer Experiment

Run repeated ANN trainings for a range of hidden-layer sizes. The experiment uses the same dataset split, normalization, model-building, early-stopping, and evaluation helpers as the main workflow. Results are returned as a DataFrame and saved to `results/ann_hidden_units_experiment.csv`.


In [ ]:
RUN_HIDDEN_UNIT_EXPERIMENT = False

if RUN_HIDDEN_UNIT_EXPERIMENT:
    experiment_results = run_hidden_unit_experiment(
        concrete_data,
        hidden_unit_range=range(71, 81),
        repeats=15,
        max_epochs=3000,
        patience=100,
        batch_size=10,
        output_path=RESULTS_DIR / "ann_hidden_units_experiment.csv",
        random_state=RANDOM_STATE,
    )
    display(experiment_results.head())
else:
    print(
        "Automated hidden-unit experiment skipped. "
        "Set RUN_HIDDEN_UNIT_EXPERIMENT = True to run the full experiment."
    )


## Conclusion and Key Findings

This notebook provides the first complete workflow for predicting compressive strength of nano-silica modified concrete with an artificial neural network. The workflow now uses descriptive feature names and a clear ANN modeling path. Further development can add a compact metrics table and compare the ANN against additional regression baselines.
